# 🏃‍♂️ PLAYHACK: Sports Injury Prediction & Biometrics Pipeline
### End-to-End Leakage-Safe Machine Learning System

This notebook provides the complete, runnable workflow for the PLAYHACK Sports Injury Prediction challenge:
1. **Multi-Source Data Ingestion & Standardization**: Fast, memory-efficient loading of wearable, activity, sleep, and session records.
2. **Strict Temporal Leakage Guard**: Validates that all features are computed exclusively within the 30-day observation window.
3. **Multi-Horizon Feature Engineering**: 70+ physiological, workload (ACWR), and sleep indicators across 7d, 14d, and 30d horizons.
4. **Task A (Injury Risk Classification)**: 5-fold `GroupKFold` cross-validation with `CalibratedClassifierCV`, optimal F1 threshold optimization, and Top-3 ensembling.
5. **Task B & Task C (Onset Day & Recovery Duration Regressions)**: Conditional regressors trained on verified injured athletes with physical boundary clipping.
6. **Leaderboards, Evaluation & Feature Attribution**: Complete benchmark leaderboard, confusion matrices, ROC curves, and feature importance plots.
7. **Model Serialization & Submission Generation**: Saves trained fold models to `outputs/models/` and generates the final `submission.csv`.

## 1. Setup, Imports & Configuration

In [ ]:
import os
import sys
import gc
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in Python path
ROOT_DIR = os.path.dirname(os.path.abspath('..')) if os.path.exists('..') else os.getcwd()
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.config import DATA_DIR, OUTPUTS_DIR, MODELS_DIR, FEATURES_PARQUET_PATH, DEFAULT_OBS_END
from src.data_loader import RawDataLoader, load_all_raw_data
from src.data_validation import DataValidator
from src.leakage_guard import leakage_guard, validate_date_boundary
from src.feature_engineering import combine_features
from src.models import TabularPreprocessor, get_classification_models, get_regression_models
from src.training import Trainer
from src.prediction import Predictor
from src.utils import set_seed, LOGGER

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

set_seed(42)
print('✅ Environment initialized successfully.')
print(f'📁 Project Root: {ROOT_DIR}')
print(f'📁 Data Directory: {DATA_DIR}')
print(f'📁 Outputs Directory: {OUTPUTS_DIR}')

## 2. Multi-Source Raw Data Loading & Standardization
Loads all raw multi-source CSV files with fast alias resolution and date standardizations without memory overhead.

In [ ]:
loader = RawDataLoader(DATA_DIR)

meta_df = loader.load_athlete_metadata()
labels_df = loader.load_train_labels()
act_df = loader.load_daily_activity()
sleep_df = loader.load_sleep_day()
sessions_df = loader.load_training_sessions()
weight_df = loader.load_weight_logs()

print('\n--- Ingestion Summary ---')
print(f'👤 Athlete Metadata:    {meta_df.shape[0]:,} rows × {meta_df.shape[1]} cols')
print(f'🎯 Training Labels:      {labels_df.shape[0]:,} rows × {labels_df.shape[1]} cols')
print(f'🏃 Daily Activity:       {act_df.shape[0]:,} rows × {act_df.shape[1]} cols')
print(f'🛌 Sleep Day Logs:       {sleep_df.shape[0]:,} rows × {sleep_df.shape[1]} cols')
print(f'🏋️ Training Sessions:    {sessions_df.shape[0]:,} rows × {sessions_df.shape[1]} cols')
print(f'⚖️ Weight Logs:          {weight_df.shape[0]:,} rows × {weight_df.shape[1]} cols')

# Preview Athlete Metadata and Targets
display_df = meta_df.merge(labels_df[['athlete_id', 'injured_in_risk_window', 'onset_day_offset', 'recovery_duration']], on='athlete_id')
display_df.head(5)

## 3. Data Quality & Target Distribution Inspection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Task A: Class Distribution
counts = labels_df['injured_in_risk_window'].value_counts()
sns.barplot(x=['Healthy (0)', 'Injured (1)'], y=counts.values, ax=axes[0], palette=['#2ca02c', '#d62728'])
axes[0].set_title(f'Task A: Injury Class Balance ({counts[1]/len(labels_df)*100:.1f}% Injured)')
axes[0].set_ylabel('Number of Athletes')

# Task B: Onset Distribution (Injured only)
inj_only = labels_df[labels_df['injured_in_risk_window'] == 1]
sns.histplot(inj_only['onset_day_offset'], bins=15, kde=True, ax=axes[1], color='#1f77b4')
axes[1].set_title('Task B: Onset Day Offset (Day 1 - 30)')
axes[1].set_xlabel('Onset Day Offset')

# Task C: Recovery Duration Distribution
sns.histplot(inj_only['recovery_duration'], bins=15, kde=True, ax=axes[2], color='#ff7f0e')
axes[2].set_title('Task C: Recovery Duration (Days)')
axes[2].set_xlabel('Recovery Days')

plt.tight_layout()
plt.show()

## 4. Multi-Horizon Feature Engineering & Leakage Guard
Computes rolling workload trends (ACWR proxies: 7d vs 30d ratios), sleep regularity, and biometrics strictly within the observation window.

In [ ]:
# Load or compute features
if os.path.exists(FEATURES_PARQUET_PATH):
    print(f'⚡ Loading precomputed features from {FEATURES_PARQUET_PATH}...')
    features_df = pd.read_parquet(FEATURES_PARQUET_PATH)
else:
    print('⚙️ Computing feature matrix from raw records...')
    raw_dict = load_all_raw_data(DATA_DIR, default_obs_end=DEFAULT_OBS_END)
    features_df = combine_features(raw_dict, obs_end=DEFAULT_OBS_END, include_labels=True)
    os.makedirs(os.path.dirname(FEATURES_PARQUET_PATH), exist_ok=True)
    features_df.to_parquet(FEATURES_PARQUET_PATH, index=False)

print(f'✅ Feature dataset shape: {features_df.shape[0]:,} athletes × {features_df.shape[1]} features')

# Inspect Workload Spike Hypothesis (ACWR)
if 'training_load_change_7d_vs_30d' in features_df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=features_df, x='injured_in_risk_window', y='training_load_change_7d_vs_30d', palette=['#2ca02c', '#d62728'])
    plt.title('Training Load Spike Ratio (7d vs 30d) vs Injury Status')
    plt.xlabel('Injured in Risk Window (0 = Healthy, 1 = Injured)')
    plt.ylabel('Workload ACWR Ratio')
    plt.show()

## 5. End-to-End Model Training & Cross-Validation Benchmarks
Runs 5-fold `GroupKFold` CV across all candidate classifiers and regressors, optimizing decision thresholds for F1-score.

In [ ]:
trainer = Trainer(features_df, n_splits=5, random_state=42)
benchmark_results = trainer.run_full_pipeline()
print('✅ Cross-validation and model training finished!')

## 6. Model Benchmark Leaderboard & Visualizations

In [ ]:
leaderboard_df = pd.DataFrame(benchmark_results['leaderboard'])
print('=== 🏆 Full Model Benchmark Leaderboard ===')
print(leaderboard_df.to_string(index=False))

# Visual Comparison of Classification F1
cls_lb = leaderboard_df[leaderboard_df['task'] == 'classification'].sort_values('score', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=cls_lb, x='score', y='model', palette='Blues_r', ax=ax)
ax.set_title('Task A: Classification OOF F1-Score Comparison')
ax.set_xlabel('F1-Score (Higher is Better)')
ax.set_ylabel('Model')
for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}", (p.get_width() + 0.005, p.get_y() + p.get_height()/2),
                va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Feature Importance & Model Explainability

In [ ]:
from src.config import FEATURE_IMPORTANCE_PATH
if os.path.exists(FEATURE_IMPORTANCE_PATH):
    imp_df = pd.read_csv(FEATURE_IMPORTANCE_PATH).head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
    plt.title('Top 15 Most Predictive Features for Sports Injury Risk', fontsize=12)
    plt.xlabel('Normalized Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 8. Test Set Batch Inference & Submission Generation
Generates the final test set predictions with 5-fold model ensembling and outputs `submission.csv`.

In [ ]:
predictor = Predictor(models_dir=MODELS_DIR)
submission_df = predictor.generate_submission(features_df, output_path=os.path.join(OUTPUTS_DIR, 'submission.csv'))

print('\n--- Final Submission Preview ---')
print(submission_df.head(10).to_string(index=False))
print(f"\n✅ Total Athletes Predicted: {len(submission_df):,}")
print(f"Predicted Injury Rate: {(submission_df['injured_in_risk_window'] == 1).mean() * 100:.2f}%")